# Author identification with RNNs

We are going to work on a corpus of classical English literature. 9 authors, 2 books per author, one file per chapter.

## Data download

In [ ]:
!git clone https://github.com/nzmonzmp/dataset-9classical-author.git

## Spacy model download

In [ ]:
!python -m spacy download en

## Imports

In [ ]:
import re
import typing

import numpy
import pathlib
import seaborn
import sklearn.model_selection
import sklearn.preprocessing
import spacy
import tensorflow.keras as keras
import tqdm.notebook

## Preprocessing

- One sentece = one example
- We replace named entities to increase the difficulty of the problem and turn it into a stylistic problem:
  - Person
  - Building
  - Organization
  - Location
  - Date
  - Currency
  - Work of art

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["tok2vec", "tagger", "attribute_ruler", "lemmatizer"])
ner_to_replace= dict(PERSON="person",
                     FAC="building" , 
                     ORG="organization", 
                     GPE="city" ,
                     LOC="lake" , 
                     DATE="date" , 
                     MONEY="dollar" , 
                     WORK_OF_ART="painting")


def get_data(directory: pathlib.Path
            ) -> typing.Tuple[typing.List[str], typing.List[str]]:
  texts = []
  authors = []
  for item in tqdm.notebook.tqdm(list(directory.glob("*/*/*.txt"))):
    author = item.parent.parent.name
    text = item.read_text(encoding="utf8")
    doc = nlp(" ".join(text.split()))
    for s in doc.sents:
      seq = [(ner_to_replace[t.ent_type_]
              if t.ent_type_ in ner_to_replace
              else t.text)
             for t in s
             if t.ent_iob_ != "I"]
      if not re.search("chapter", seq[0], re.IGNORECASE):
        texts.append((" ".join(seq)))
        authors.append(author)
  return texts, authors

texts, authors = get_data(pathlib.Path("dataset-9classical-author"))
print(len(texts), len(authors))


label_encoder = sklearn.preprocessing.LabelEncoder()
y = label_encoder.fit_transform(authors)
print(y)
X_train_raw, X_test_raw, y_train, y_test = \
    sklearn.model_selection.train_test_split(texts, y, test_size=0.3)

In [ ]:
print(len(X_train_raw), y_train.shape)

## Turning sentences into sequences

### Preparing the vocabulary dictionary

[`tensorflow.keras.preprocessing.text.Tokenizer`](https://keras.io/api/preprocessing/text/#tokenizer-class) is very helpful for that.

In [ ]:
tokenizer_obj = keras.preprocessing.text.Tokenizer()
tokenizer_obj.fit_on_texts(X_train_raw)

In [ ]:
vocab_size = len(tokenizer_obj.word_index) + 1
max_length = max(len(s.split()) for s in X_train_raw)
print(f"Vocabulary size: {vocab_size}")
print(f"Size of the lengthiest sentence: {max_length}")

### Using the vocabulary to produce a matrix of indices to represent sentences

Let's combine [`tensorflow.keras.preprocessing.text.Tokenizer`](https://keras.io/api/preprocessing/text/#tokenizer-class) & [`tensorflow.keras.preprocessing.sequence.pad_sequences`](https://keras.io/api/preprocessing/timeseries/#padsequences-function) to obtain a matrix representation of our input sentences.

In [ ]:
max_length = 150

X_train_tokens = tokenizer_obj.texts_to_sequences(X_train_raw)
X_test_tokens = tokenizer_obj.texts_to_sequences(X_test_raw)

X_train_pad = keras.preprocessing.sequence.pad_sequences(
    X_train_tokens, maxlen=max_length, truncating="pre")
X_test_pad = keras.preprocessing.sequence.pad_sequences(
    X_test_tokens, maxlen=max_length, truncating="pre")

In [ ]:
print(f"Input shape: {X_train_pad.shape}")
print(f"First example: {X_train_pad[0]}")

## Model design

We are going to use a model comprised of:
  - An embedding layer that projects each word in a 300-dimensional space
  - A bi-directional GRU layer:
    - with `64` neurons
    - that uses orthogonal initialization for its weight matrices
    - 20% dropout
  - A dense layer
  - A softmax activation function
  - Cross-entropy as a loss
  - Adam optimizer
  - Accuracy as a metric

In [ ]:
EMBEDDING_DIM = 300

model = keras.models.Sequential()
model.add(keras.layers.InputLayer(input_shape=(max_length,)))
model.add(keras.layers.Embedding(vocab_size, EMBEDDING_DIM))
model.add(keras.layers.Bidirectional(
    keras.layers.GRU(64,
                     kernel_initializer="orthogonal",
                     recurrent_initializer="orthogonal",
                     dropout=0.2,
                     recurrent_dropout=0.2)))
model.add(keras.layers.Dense(9, activation="softmax"))
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])
model.summary()

## Training

In [ ]:
model.fit(X_train_pad,
          y_train,
          batch_size=1024,
          epochs=10,
          validation_split=0.3)

## Evaluation

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test_pad, y_test)
print(f"Accuracy on test data: {test_accuracy:.3f}")

We can notice overfitting: train $\approx$ 75% vs validation $\approx$ 50%.

## Pre-trained word embeddings usage

To limit overfitting, we will use pretrained and frozen word embeddings.

In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

In [ ]:
def glove_path(embedding_dim: int) -> pathlib.Path:
  if embedding_dim in {50, 100, 200, 300}:
    return pathlib.Path("glove.6B.{}d.txt".format(embedding_dim))
  else:
    raise ValueError("embedding_dim must be in {50, 100, 200, 300}")

In [ ]:
embedding_matrix = numpy.zeros((vocab_size, EMBEDDING_DIM))
found = 0
with glove_path(EMBEDDING_DIM).open() as fh:
  for line in fh:
    values = line.split(" ")
    word = values[0]
    if word in tokenizer_obj.word_index:
      found += 1
      coeffs = numpy.array(values[1:], dtype="float32")
      embedding_matrix[tokenizer_obj.word_index[word]] = coeffs

print(f"Usage of {found} pre-trained embeddings, for a total of {vocab_size} "
      "words in the vocabulary")

embedding_layer = keras.layers.Embedding(vocab_size,
                                         EMBEDDING_DIM,
                                         weights=[embedding_matrix],
                                         input_length=max_length,
                                         trainable=False)

In [ ]:
model = keras.models.Sequential()
model.add(embedding_layer)
model.add(keras.layers.Bidirectional(
    keras.layers.GRU(64,
                     kernel_initializer="orthogonal",
                     recurrent_initializer="orthogonal",
                     dropout=0.2,
                     recurrent_dropout=0.2)))
model.add(keras.layers.Dense(9, activation ="softmax"))

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])
model.summary()

In [ ]:
model.fit(X_train_pad,
          y_train,
          batch_size=1024,
          epochs=20,
          validation_split=0.3)

In [ ]:
model.evaluate(X_test_pad, y_test)

## Confusion matrix

In [ ]:
y_pred = model.predict_classes(X_test_pad)

In [ ]:
conf_mat = sklearn.metrics.confusion_matrix(y_pred, y_test, normalize="true")
seaborn.heatmap(conf_mat,
                vmin=0,
                vmax=1,
                cmap="rocket_r")